# Study 914 — Securities-Lending Offset 📚

**Index funds lend their shares to short sellers. Does the money come back to you?**

The pitch is seductive and, in the sponsors' own reports, true: an index fund lends the
shares it holds, collects a fee, and credits most of it back to the fund. In hard-to-borrow
corners — small caps, emerging markets — that revenue is supposed to offset a real slice of
the expense ratio, so the fund's **tracking difference** should land *better* than `−ER`.
"Our EM fund is effectively free," goes the brochure version.

We test it on **five same-asset-class ETF pairs** — SPY/IVV, EEM/IEMG, EFA/IEFA, VEA/IEFA,
IWM/IJR — using daily **total-return** closes aggregated to monthly, as-of 2026-06-30.

*Real numbers below are the frozen headline (`docs/results.md`, Fingerprint `d080dc025da8`).
The live cells run the offline synthetic control only.*


## 1. The idea, and the one number that decides it

Two funds hold nearly the same basket. One charges more. If nothing else were going on, the dearer fund should trail the cheaper one by *exactly* the fee difference. So take the realised gap and add the fee gap back:

> **residual = realised return gap + fee gap**

If that residual is **zero**, fees explained everything. If it is **positive** for the dearer fund, that fund handed back something extra — which is what lending revenue is supposed to look like from the outside.

> 🔬 *For the quants:* the residual is measured on month-end log total returns (daily differencing between two funds is dominated by dividend-date mismatch), with an HAC *t* and a 6-month-block bootstrap CI. Expense ratios are a labelled **assumption**, not tape, and are swept.

In [1]:
pairs = [('SPY', 'IVV', 'US large cap', '2000-06', '2026-06', 313, -1.4, 6.5, 5.1, 1.04, -5.3, 15.1, 0.52, 20, 1.002), ('EEM', 'IEMG', 'Emerging markets', '2012-11', '2026-06', 164, -58.0, 61.0, 3.0, 0.11, -50.1, 56.2, 1.24, 67, 1.015), ('EFA', 'IEFA', 'Developed ex-US', '2012-11', '2026-06', 164, -21.4, 26.0, 4.6, 0.24, -33.6, 43.4, 0.82, 44, 0.993), ('VEA', 'IEFA', 'Developed ex-US (cross-sponsor)', '2012-11', '2026-06', 164, 52.7, -4.0, 48.7, 1.14, -25.1, 137.0, 1.26, 68, 1.022), ('IWM', 'IJR', 'US small cap', '2000-06', '2026-06', 313, -127.3, 13.0, -114.3, -1.5, -264.3, 30.0, 3.75, 147, 1.013)]
hdr = 'pair            class                     resid    t      95% CI          TE    floor'
print(hdr); print('-' * len(hdr))
for a, b, k, s, e, n, dr, der, res, t, lo, hi, te, fl, beta in pairs:
    print('%-4s-%-5s %-25s %+7.1f %+6.2f  [%+7.1f,%+7.1f] %5.2f%% %4dbp'
          % (a, b, k[:25], res, t, lo, hi, te, fl))

pair            class                     resid    t      95% CI          TE    floor
-------------------------------------------------------------------------------------
SPY -IVV   US large cap                 +5.1  +1.04  [   -5.3,  +15.1]  0.52%   20bp
EEM -IEMG  Emerging markets             +3.0  +0.11  [  -50.1,  +56.2]  1.24%   67bp
EFA -IEFA  Developed ex-US              +4.6  +0.24  [  -33.6,  +43.4]  0.82%   44bp
VEA -IEFA  Developed ex-US (cross-sp   +48.7  +1.14  [  -25.1, +137.0]  1.26%   68bp
IWM -IJR   US small cap               -114.3  -1.50  [ -264.3,  +30.0]  3.75%  147bp


## 2. The tight pairs say: fees explain everything

For the three pairs that genuinely track the same thing, the residual is **+3.0, +4.6 and +5.1 basis points a year** — with *t*-statistics of 0.11, 0.24 and 1.04. That is nothing. EEM trailed IEMG by 58 bp/yr; its fee disadvantage is 61 bp/yr. The gap *is* the fee, to within three basis points.

Pool the three and you get **+3.7 bp/yr (*t* = +0.31)**, CI [-20.0, +27.2].

## 3. The clean experiment nobody set up on purpose

**SPY cannot lend its shares.** It was built in 1993 as a unit investment trust, and that structure forbids securities lending. **IVV can, and does.** Same index, same 500 stocks, tightest tracking on the desk.

If lending revenue reached shareholders in any visible way, IVV should beat SPY by *more* than the fee gap. Over 26 years **SPY trailed IVV by 1.4 bp/yr** — against a fee gap that is somewhere between 0 and 6.5 bp/yr depending on which year's fee schedule you use (IVV's fee fell from 0.0945% to 0.03% over exactly this window; see §4). The two funds ran neck and neck.

That is a **bound**, not a measurement: any IVV lending advantage visible in returns is within a few basis points a year, and the bootstrap CI caps it at **≈15 bp/yr**. Which way the small leftover points is decided by the fee assumption, not by the tape.

In [2]:
spy = dict(zip(['a', 'b', 'klass', 'start', 'end', 'n', 'drift', 'dER',
                'resid', 't', 'ci_lo', 'ci_hi', 'te', 'floor', 'beta'], ('SPY', 'IVV', 'US large cap', '2000-06', '2026-06', 313, -1.4, 6.5, 5.1, 1.04, -5.3, 15.1, 0.52, 20, 1.002)))
print('SPY (cannot lend) vs IVV (lends): {start} -> {end}, {n} months'.format(**spy))
print('  realised gap  %+7.1f bp/yr   (SPY minus IVV) <- this is the tape'
      % spy['drift'])
print('  residual under TODAY\'s fees      (dER {dER:+.1f} bp): {resid:+.1f} bp/yr'
      '  HAC t = {t:+.2f}'.format(**spy))
print('  residual under 2000-era fees     (dER  +0.0 bp): -1.4 bp/yr'
      '  HAC t = -0.28')
print('  95% CI on the headline: [{ci_lo:+.1f}, {ci_hi:+.1f}] bp/yr'.format(**spy))
print()
print('  -> the SIGN flips with the fee assumption; only the MAGNITUDE is the tape\'s.')
print('  -> bound: any IVV passthrough visible in returns <= 15 bp/yr.')

SPY (cannot lend) vs IVV (lends): 2000-06 -> 2026-06, 313 months
  realised gap     -1.4 bp/yr   (SPY minus IVV) <- this is the tape
  residual under TODAY's fees      (dER +6.5 bp): +5.1 bp/yr  HAC t = +1.04
  residual under 2000-era fees     (dER  +0.0 bp): -1.4 bp/yr  HAC t = -0.28
  95% CI on the headline: [-5.3, +15.1] bp/yr

  -> the SIGN flips with the fee assumption; only the MAGNITUDE is the tape's.
  -> bound: any IVV passthrough visible in returns <= 15 bp/yr.


## 4. The catch that decides the sign: fees were cut

Every one of these residuals is a *difference of two fees*, and we do not have a tape of fees — we have today's published numbers. That matters here more than anywhere, because **the cheap fund in each pair had its fee cut hard during the sample** (IVV 0.0945% → 0.03%, IEMG 0.18% → 0.09%, IEFA 0.14% → 0.07%) while the expensive one barely moved.

Charging the cheap fund its *current* fee for its whole history therefore makes the fee gap look bigger than it was, which pushes every residual **up** — in exactly the direction that makes a lending offset look absent. Undo that, and all three tight-pair residuals cross zero:

In [3]:
rows = [('SPY-IVV', 'IVV', [('earliest', 0.0945, 0.0, -1.4, -0.28), ('mid', 0.065, 2.9, 1.6, 0.32), ('today', 0.03, 6.5, 5.1, 1.04)]), ('EEM-IEMG', 'IEMG', [('earliest', 0.18, 52.0, -6.0, -0.22), ('mid', 0.13, 57.0, -1.0, -0.03), ('today', 0.09, 61.0, 3.0, 0.11)]), ('EFA-IEFA', 'IEFA', [('earliest', 0.14, 19.0, -2.4, -0.13), ('mid', 0.09, 24.0, 2.6, 0.13), ('today', 0.07, 26.0, 4.6, 0.24)])]
print('pair       cheap leg   fee assumption      dER      residual      t')
for name, cheap, sweep in rows:
    for label, er_b, der, res, t in sweep:
        print('%-10s %-11s %-9s %.4f%%  %+7.1f bp  %+7.1f bp  %+6.2f'
              % (name, cheap, label, er_b, der, res, t))
print()
print('The residual is smaller than the error in the fee input. So the honest')
print('output is a MAGNITUDE (|residual| <= ~6 bp/yr), with no direction attached.')

pair       cheap leg   fee assumption      dER      residual      t
SPY-IVV    IVV         earliest  0.0945%     +0.0 bp     -1.4 bp   -0.28
SPY-IVV    IVV         mid       0.0650%     +2.9 bp     +1.6 bp   +0.32
SPY-IVV    IVV         today     0.0300%     +6.5 bp     +5.1 bp   +1.04
EEM-IEMG   IEMG        earliest  0.1800%    +52.0 bp     -6.0 bp   -0.22
EEM-IEMG   IEMG        mid       0.1300%    +57.0 bp     -1.0 bp   -0.03
EEM-IEMG   IEMG        today     0.0900%    +61.0 bp     +3.0 bp   +0.11
EFA-IEFA   IEFA        earliest  0.1400%    +19.0 bp     -2.4 bp   -0.13
EFA-IEFA   IEFA        mid       0.0900%    +24.0 bp     +2.6 bp   +0.13
EFA-IEFA   IEFA        today     0.0700%    +26.0 bp     +4.6 bp   +0.24

The residual is smaller than the error in the fee input. So the honest
output is a MAGNITUDE (|residual| <= ~6 bp/yr), with no direction attached.


## 5. The big numbers are the wrong kind of big

Two pairs *do* show large residuals — and both are red herrings:

- **IWM − IJR: −114 bp/yr.** Those are Russell 2000 and S&P SmallCap 600, two different index families with a well-documented quality gap. The fee gap is 13 bp. A number nine times too large, in a pair that does not share a benchmark, is telling you about index construction, not about lending.
- **VEA − IEFA: +49 bp/yr.** FTSE and MSCI disagree about which countries count as developed. Korea and Canada sit in one and not the other.

> 🔬 *For the quants:* neither clears |*t*| = 2 over the full sample, and neither is stable across the 2020 era split.

## 6. Why the tape could never have settled this

Here is the uncomfortable arithmetic. Two same-class funds still drift apart by 0.5% to 3.8% a year at random. Over the sample we have, that random drift means the smallest effect we could ever call statistically real is **20 to 147 basis points a year**.

And the thing we are hunting — net lending revenue credited to a broad index fund — is worth perhaps **1 to 15 basis points a year**.

So this study cannot *measure* the offset. It can only **bound** it. That bound is the honest deliverable, and it is a useful one: whatever the lending desks earn, what reaches you as measurable *relative* return is smaller than roughly 6 bp/yr on every same-sponsor pair, on any fee assumption.

## 7. Live check — the machinery works (offline synthetic)

Before believing a zero, check the detector can find a one. On a *simulated* pair where we plant 60 bp/yr of lending passthrough by hand, the same estimator must find it; on a simulated pair with no passthrough it must stay silent.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from lending_offset import data, strategy as st
planted = st.synthetic_detect(*data.synthetic_daily(signal_strength=1.0, seed=914))
print('planted +%.0f bp/yr -> recovered %+.1f bp/yr  (t = %+.2f)'
      % (planted['planted_bp'], planted['residual_bp'], planted['t_hac']))
nulls = [st.synthetic_detect(*data.synthetic_daily(signal_strength=0.0, seed=914 + s))
         for s in range(8)]
res = np.array([n['residual_bp'] for n in nulls])
ts  = np.array([n['t_hac'] for n in nulls])
print('null x8            -> mean %+.1f bp/yr (sd %.1f), fires |t|>=2 in %d/8'
      % (res.mean(), res.std(ddof=1), (abs(ts) >= 2).sum()))

planted +60 bp/yr -> recovered +63.1 bp/yr  (t = +4.89)


null x8            -> mean +2.7 bp/yr (sd 13.9), fires |t|>=2 in 0/8


## 8. Could you at least trade the fee gap?

Barely. Long IEMG, short EEM harvests the 61 bp fee difference: **+47 bp/yr** gross, Sharpe +0.38. Then the short leg pays borrow. At 25 bp of borrow it is +22 bp. At **50 bp of borrow — which is what an EM ETF actually costs to borrow — it is −3 bp.** Gone.

The practical lesson is the boring one: if you want the cheap fund's return, **own the cheap fund**. Do not try to short the expensive one to capture the difference.

## Verdict

- **Signal — None.** The lending offset does not show up in returns. The tight pairs' residuals are **+3.0 / +4.6 / +5.1 bp/yr** (|*t*| ≤ 1.04), pooled **+3.7 bp/yr (*t* = +0.31)** — and **-0.6 bp/yr** if the funds are charged the fees they actually charged at the start of the sample. The one structural control — SPY, which legally *cannot* lend — ran level with IVV. The two large residuals are index-composition wedges. Honest caveats: the test is **underpowered by design** (noise floor 20–147 bp/yr against a 1–15 bp effect); **the fee assumption, not the tape, decides the residual's sign**, so only its magnitude is reported; and every fund here is a **survivor**.
- **Tradability — Mirage.** No lending spread to bank. The fee gap itself is tradable only until borrow reaches ~50 bp/yr, at which point it is exactly zero.
- **What you should take away.** The offset is real in the annual reports and invisible in the tape. Choose funds on the fee you can see, not on the lending revenue you are promised.